# Calendar Dimension Table

Builds the standard calendar (date dimension) table `adb_medsync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
%sql
-- Standard calendar dimension: one row per day, 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_medsync.silver.calendar
COMMENT 'Standard calendar dimension table, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)                 AS date_key,
  date                                                         AS date,
  -- Year
  year(date)                                                   AS year,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0
       THEN true ELSE false END                                AS is_leap_year,
  CAST(date_trunc('YEAR', date) AS DATE)                       AS first_day_of_year,
  make_date(year(date), 12, 31)                                AS last_day_of_year,
  -- Half year
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END                 AS half_year,
  concat('H', CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END)    AS half_year_name,
  -- Quarter
  quarter(date)                                                AS quarter,
  concat('Q', quarter(date))                                   AS quarter_name,
  concat(year(date), '-Q', quarter(date))                      AS year_quarter,
  CAST(date_trunc('QUARTER', date) AS DATE)                    AS first_day_of_quarter,
  date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS last_day_of_quarter,
  -- Month
  month(date)                                                  AS month,
  date_format(date, 'MMMM')                                    AS month_name,
  date_format(date, 'MMM')                                     AS month_short_name,
  date_format(date, 'yyyy-MM')                                 AS year_month,
  year(date) * 100 + month(date)                               AS year_month_key,
  CAST(date_trunc('MONTH', date) AS DATE)                      AS first_day_of_month,
  last_day(date)                                               AS last_day_of_month,
  day(last_day(date))                                          AS days_in_month,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(date)                                             AS week_of_year,
  extract(YEAROFWEEK FROM date)                                AS iso_week_year,
  concat(extract(YEAROFWEEK FROM date), '-W', lpad(weekofyear(date), 2, '0')) AS iso_year_week,
  CAST(ceil(day(date) / 7.0) AS INT)                           AS week_of_month,
  date_sub(date, weekday(date))                                AS week_start_date,
  date_add(date_sub(date, weekday(date)), 6)                   AS week_end_date,
  -- Day
  day(date)                                                    AS day_of_month,
  dayofyear(date)                                              AS day_of_year,
  weekday(date) + 1                                            AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(date, 'EEEE')                                    AS day_name,
  date_format(date, 'EEE')                                     AS day_short_name,
  -- Flags
  weekday(date) >= 5                                           AS is_weekend,
  weekday(date) < 5                                            AS is_weekday,
  date = CAST(date_trunc('MONTH', date) AS DATE)               AS is_month_start,
  date = last_day(date)                                        AS is_month_end,
  date = CAST(date_trunc('QUARTER', date) AS DATE)             AS is_quarter_start,
  date = date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS is_quarter_end,
  (month(date) = 1 AND day(date) = 1)                          AS is_year_start,
  (month(date) = 12 AND day(date) = 31)                        AS is_year_end
FROM dates
ORDER BY date

In [0]:
%sql
-- Validation: row count, date range, uniqueness of keys
SELECT
  count(*)                         AS row_count,
  count(DISTINCT date_key)         AS distinct_date_keys,
  datediff(max(date), min(date)) + 1 AS expected_days,
  min(date)                        AS min_date,
  max(date)                        AS max_date,
  sum(CAST(is_leap_year AND month = 2 AND day_of_month = 29 AS INT)) AS leap_days
FROM adb_medsync.silver.calendar

In [0]:
%sql
SELECT * FROM adb_medsync.silver.calendar
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-05'
ORDER BY date